In [0]:
import pyspark.sql.functions as f

data = [
    {"ID": 1, "Name": "John"},
    {"ID": 2, "Name": "Isabel"},
    {"ID": 3, "Name": "Bart"}
]

df = spark.createDataFrame(data)
df.show()

In [0]:
# From list of tuples
data = [(1, 'John'), (2, 'Isabel'), (3, 'Bart')]
df = spark.createDataFrame(data, ['ID', 'Name'])
df.show()

In [0]:
import pandas as pd
pandas_df = pd.DataFrame({'ID': [1,2,3], 'Name': ['John','Isabel','Bart']})
spark_df = spark.createDataFrame(pandas_df)
spark_df.show()

In [0]:
df_back = spark_df.toPandas()
df_back.show()

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
~/.ipykernel/2687/command-4902806220291815-3986198225 in ?()
      1 df_back = spark_df.toPandas()
----> 2 df_back.show()

/databricks/python/lib/python3.11/site-packages/pandas/core/generic.py in ?(self, name)
   5898             and name not in self._accessors
   5899             and self._info_axis._can_hold_identifiers_and_holds_name(name)
   5900         ):
   5901             return self[name]
-> 5902         return object.__getattribute__(self, name)

AttributeError: 'DataFrame' object has no attribute 'show'

In [0]:
df_back.head()

In [0]:
# IMPORTANT - SPECIFY THE CATALOG FIRST!
#df.write.saveAsTable('users')
#df.write.saveAsTable('my_catalog.my_database.users')

In [0]:
spark.read.table('users').show()

In [0]:
import requests

url = "https://raw.githubusercontent.com/afonso-pereira/spark-data/refs/heads/main/data/people-100.csv"
volume_path = "/Volumes/workspace/default/class/people-100.csv"

with open(volume_path, "wb") as f:
    f.write(requests.get(url).content)

df = spark.read.csv(volume_path, header=True, inferSchema=True)
df.display()

In [0]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .load("/Volumes/workspace/default/class/people-100.csv")
)
df.display()

In [0]:
import requests, zipfile, os

# Download and extract to volume
volume_path = "/Volumes/workspace/default/class"
zip_path = f"{volume_path}/orders-data.zip"

with open(zip_path, "wb") as f:
    f.write(requests.get("https://github.com/afonso-pereira/spark-data/raw/refs/heads/main/data/orders-data.zip").content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(f"{volume_path}/orders-data")

In [0]:
df_orders = spark.read.csv(
    "/Volumes/workspace/default/class/orders-data/orders-data/orders.csv",
    header=True, inferSchema=True
)

In [0]:
df_orders.printSchema()

In [0]:
df_orders.count()

In [0]:
df_orders.describe().display()

In [0]:
df_orders.summary().display()

In [0]:
df_renamed = df_orders.select(
    f.col('Customer ID').alias('customer_id'),
    f.col('Customer Status').alias('customer_status'),
    f.col('Date Order was placed').alias('placing_date'),
    f.col('Delivery Date').alias('delivery_date'),
    f.col('Order ID').alias('order_id'),
    f.col('Product ID').alias('product_id'),
    f.col('Quantity Ordered').alias('amount'),
    f.col('Total Retail Price for This Order').alias('revenue'),
    f.col('Cost Price Per Unit').alias('cost_per_unit')
)
df_renamed.show(3)

In [0]:
df_std = (
    df_renamed
    .withColumn(
        'customer_status_new',
        f.lower(f.col('customer_status'))
    )
    .drop('customer_status')
    .withColumnRenamed('customer_status_new', 'customer_status')
)
df_std.groupBy('customer_status').count().show()

In [0]:
# Remove exact duplicate rows
df_orders = df_renamed.dropDuplicates()

# Remove duplicates based on specific columns only
df_orders = df_orders.dropDuplicates(subset=['order_id'])

df_orders.count()

In [0]:
# Before
df_renamed.describe().display() 

In [0]:
df_cleaned = (
    df_renamed
    .fillna(1, subset=['amount'])
    .dropna(subset=['placing_date'])
)
df_cleaned.describe().display()

In [0]:
df_cleaned = (
    df_cleaned
    .withColumn('customer_status', f.lower(f.col('customer_status')))
)

In [0]:
months_mapping = {
    'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
    'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
    'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
}

df_orders_date = df_cleaned
for abbr, nr in months_mapping.items():
    df_orders_date = (
        df_orders_date
        .withColumn('placing_date',  f.regexp_replace('placing_date',  abbr, nr))
        .withColumn('delivery_date', f.regexp_replace('delivery_date', abbr, nr))
    )

df_orders_date = (
    df_orders_date
    .withColumn('placing_date',  f.to_date(f.col('placing_date'),  'dd-MM-yy'))
    .withColumn('delivery_date', f.to_date(f.col('delivery_date'), 'dd-MM-yy'))
    .withColumn('days_to_delivery',   f.datediff(f.col('delivery_date'), f.col('placing_date')))
    .withColumn('order_month',        f.month('placing_date'))
    .withColumn('order_year',         f.year('placing_date'))
    .withColumn('order_day_of_week',  f.dayofweek('placing_date'))
)

df_orders_date.select('placing_date','delivery_date','days_to_delivery','order_month','order_year','order_day_of_week').show(3)

In [0]:
df_feat = (
    df_orders_date
    .withColumn('profit',
        f.col('revenue') - (f.col('cost_per_unit') * f.col('amount')))
    .withColumn('delivery_speed',
        f.when(f.col('days_to_delivery') <= 1, 'Fast')
         .when(f.col('days_to_delivery') <= 3, 'Medium')
         .otherwise('Slow'))
)

df_feat.select('days_to_delivery', 'delivery_speed', 'profit').show(5)

In [0]:
df_products = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .load("/Volumes/workspace/default/class/orders-data/orders-data/product-supplier.csv")
)

df_products.display()

In [0]:
new_cols = [c.lower().replace(' ', '_') for c in df_products.columns]
df_products = df_products.toDF(*new_cols)

df_products.printSchema()

In [0]:
df_joined = (
    df_feat.join(
        df_products,
        on=['product_id'],
        how='left'
    )
)
df_joined.printSchema()
df_joined.select('order_id', 'product_name', 'product_category').show(4)

In [0]:
# isin() = SQL IN
df_weekends = df_joined.filter(
    f.col('order_day_of_week').isin([1, 7])
)
df_weekends.count()

In [0]:

# Multiple conditions — parentheses are REQUIRED
df_shoes_weekend = df_joined.filter(
    (f.col('product_category') == 'Shoes') &
    (f.col('order_day_of_week').isin([1, 7]))
)

df_shoes_weekend.count()

In [0]:
# Option 1: distinct() then count()
df_joined.select('customer_id').distinct().count()


In [0]:
# Option 2: countDistinct()
df_joined.select(f.countDistinct('customer_id')).show()

In [0]:
# countDistinct inside groupBy
df_joined.groupBy('order_year').agg(
    f.countDistinct('customer_id').alias('unique_customers')
).show()

In [0]:
(
    df_joined
    .groupBy('customer_status')
    .agg(
        f.avg('days_to_delivery').alias('avg_delivery_days'),
        f.sum('revenue').alias('total_revenue'),
        f.count('order_id').alias('nr_orders')
    )
    .sort(f.desc('total_revenue'))
).show()

In [0]:
# collect_list: aggregate values into an array per group
(
    df_joined
    .groupBy('product_id')
    .agg(f.collect_list('delivery_date').alias('delivery_dates'))
).display()

# stddev: standard deviation of profit per supplier
(
    df_joined
    .groupBy('supplier_id')
    .agg(f.stddev('profit').alias('profit_stddev'))
).display()

In [0]:
pivot_df = (
    df_joined
    .groupBy('order_year')
    .pivot('product_category')
    .agg(f.count('order_id'))
)
pivot_df.display()

In [0]:
df_feat.write.csv(
    '/Volumes/workspace/default/class/orders_preprocessed.csv',
    header=True
)